# Compute and plot atmospheric forecast-period bias metrics

Read model ensembles and references only for missing/forced products, cache metrics, and plot the cached maps.

## 1. Libraries


In [ ]:
from pathlib import Path
import sys


## 2. User setup: paths


In [ ]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# =============================================================================
from pathlib import Path
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
WORKFLOW_NAME = "fcst_atm_bias"


## 3. Project setup


In [ ]:
# Derived paths: built from the parameters above.
import sys
from pathlib import Path

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME, output_root=DIAGNOSTIC_OUTPUT_ROOT
)


## Notes: available variables, metrics, and periods

This notebook handles the **forecast period** only (`run_segment="fc"`).
It is intentionally separate from the DA-analysis-period workflow so their experiment
sets, periods, caches, and figures cannot be mixed.

Available atmospheric variables:

| Variable | Reference | Units |
|---|---|---|
| `FLUT` | CERES-OAFlux | W m-2 |
| `PRECT` | GPCP | mm day-1 |
| `TMQ` | ERA5 | kg m-2 |
| `TS` | ERA5 | degC |
| `TREFHT` | ERA5 | degC |
| `PSL` | ERA5 | hPa |

Available metrics:

| Metric | Meaning |
|---|---|
| `bias` | Ensemble-mean model minus reference, averaged over the selected period |
| `rmse` | Root-mean-square member error over ensemble and time |
| `spread` | Ensemble standard deviation averaged over time |
| `crps` | Continuous ranked probability score averaged over time |

Available experiment groups:

- `S0`: analysis period `201112`; forecast period `201201–201202`.
- `S1`: analysis period `201205`; forecast period `201206–201207`.

Only experiments with a configured `fc` run are available. Use
`available_experiments(SEASON, RUN_SEGMENT)` after the imports to list them.
Each experiment/variable/period has a request-keyed NetCDF cache. Existing
products are reused unless `force_compute=True`. Set `N_BOOTSTRAP=0` to skip
confidence intervals; bias hatching is drawn only when cached intervals exist.


## 4. User setup: diagnostic selections


In [ ]:
REFERENCE_DATA_ROOT = INPUT_DATA_ROOT / "reference"
CACHE_DIR = WORKFLOW_DATA_DIR / "bias_metrics"
FIGURE_DIR = WORKFLOW_FIGURE_DIR / "bias_metrics"

RUN_SEGMENT = "fc"  # Fixed by this workflow: da=analysis, fc=forecast.
COMPONENT = "atm"
SEASON = "S0"
EXPERIMENTS = ["CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"]
SELECTED_VARIABLES = ["FLUT", "PRECT", "TMQ", "TS", "TREFHT", "PSL"]
METRICS_TO_PLOT = ["bias", "rmse", "spread", "crps"]

LAT_RANGE = (-90, 90)
LON_RANGE = (-180, 180)
N_BOOTSTRAP = 100
RANDOM_SEED = 42
MAX_MEMBERS = None  # Set a small integer only for a quick test.

force_compute = False
RUN_WORKFLOW = False  # Review selections, then set True.


## 5. Workflow imports and availability


In [ ]:
from util.bias_metrics import (
    METRICS,
    available_experiments,
    plot_cached_bias_metrics,
    run_bias_metrics,
)

print("Available experiments:", available_experiments(SEASON, RUN_SEGMENT))
print("Available metrics:", list(METRICS))


## 6. Read, process, and cache


In [ ]:
if RUN_WORKFLOW:
    run_bias_metrics(
        data_root=INPUT_DATA_ROOT,
        cache_dir=CACHE_DIR,
        season=SEASON,
        run_segment=RUN_SEGMENT,
        component=COMPONENT,
        experiments=EXPERIMENTS,
        variables=SELECTED_VARIABLES,
        reference_root=REFERENCE_DATA_ROOT,
        lat_range=LAT_RANGE,
        lon_range=LON_RANGE,
        n_bootstrap=N_BOOTSTRAP,
        random_seed=RANDOM_SEED,
        max_members=MAX_MEMBERS,
        force_compute=force_compute,
    )


## 7. Plot, save, and show


In [ ]:
if RUN_WORKFLOW:
    figures = plot_cached_bias_metrics(
        data_root=INPUT_DATA_ROOT,
        cache_dir=CACHE_DIR,
        season=SEASON,
        run_segment=RUN_SEGMENT,
        component=COMPONENT,
        experiments=EXPERIMENTS,
        variables=SELECTED_VARIABLES,
        figure_dir=FIGURE_DIR,
        metrics=METRICS_TO_PLOT,
    )
    print(f"Wrote {len(figures)} figures to {FIGURE_DIR}")
